In [2]:
# -*- coding: utf-8 -*-
# 【CP2-15 缓存】CachePolicy + InMemoryCache 的结果复用与 TTL 失效
# 文件：CP2/15_ache.ipynb
# 作用：本 Cell 演示 【CP2-15 缓存】CachePolicy + InMemoryCache 的结果复用与 TTL 失效 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

from langgraph.cache.memory import InMemoryCache
from langgraph.constants import START, END
import time
from _operator import add

from langgraph.graph import StateGraph
from langgraph.types import CachePolicy  # 缓存策略：TTL 控制结果复用时长
from loguru import logger

from typing import Annotated, TypedDict


class OverAllState(TypedDict):
    user: str
    invoke_counts:Annotated[int,add]

def node_a(state:OverAllState) -> OverAllState:
    logger.info(f"node_a被调用 user:{state['user']}")  # 日志埋点：标记节点开始
    time.sleep(3)
    logger.info(f"node_a调用完成 user:{state['user']}")  # 日志埋点：标记节点结束
    return {
        "invoke_counts": 1
    }

builder = StateGraph(state_schema=OverAllState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node("node_a",node_a,cache_policy=CachePolicy(ttl=10))  # TTL=10 秒：10 秒内同输入复用缓存  # 注册节点并绑定缓存策略
builder.add_edge(START,"node_a")  # 起点扇出
builder.add_edge("node_a",END)  # 汇入终点

graph = builder.compile(cache = InMemoryCache())  # 挂载内存缓存后端；生产可用 Sqlite/Redis 后端
# 命中：同输入在 TTL 内直接返回缓存；超时后重新执行

logger.info("首次调用")
logger.info("运行结果{}\n\n",graph.invoke({"user":"test","invoke_counts":0}))  # 触发图执行：传入初始 State + config

logger.info("再次调用")
logger.info("运行结果{}\n\n",graph.invoke({"user":"test","invoke_counts":0}))  # 触发图执行：传入初始 State + config

time.sleep(11)
logger.info("再次调用")
logger.info("运行结果{}\n\n",graph.invoke({"user":"test","invoke_counts":0}))  # 触发图执行：传入初始 State + config


2026-08-06 02:11:05.616 | INFO     | __main__:<module>:32 - 首次调用
2026-08-06 02:11:05.621 | INFO     | __main__:node_a:18 - node_a被调用 user:test
2026-08-06 02:11:08.623 | INFO     | __main__:node_a:20 - node_a调用完成 user:test
2026-08-06 02:11:08.625 | INFO     | __main__:<module>:33 - 运行结果{'user': 'test', 'invoke_counts': 1}


2026-08-06 02:11:08.626 | INFO     | __main__:<module>:35 - 再次调用
2026-08-06 02:11:08.628 | INFO     | __main__:<module>:36 - 运行结果{'user': 'test', 'invoke_counts': 1}


2026-08-06 02:11:19.629 | INFO     | __main__:<module>:39 - 再次调用
2026-08-06 02:11:19.631 | INFO     | __main__:node_a:18 - node_a被调用 user:test
2026-08-06 02:11:22.633 | INFO     | __main__:node_a:20 - node_a调用完成 user:test
2026-08-06 02:11:22.634 | INFO     | __main__:<module>:40 - 运行结果{'user': 'test', 'invoke_counts': 1}


